In [ ]:
# Run once per Spark session (driver environment)
%pip install --upgrade pip
%pip install "openai==1.14.3" "pydantic==2.6.4" "typing_extensions>=4.9.0,<5"
%pip install "azure-ai-textanalytics" "azure-identity"

print("✅ Driver dependencies installed")

In [2]:
from notebookutils import mssparkutils
from azure.identity import ClientSecretCredential
from azure.ai.textanalytics import TextAnalyticsClient
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
from pyspark.sql.functions import col
import com.microsoft.spark.sqlanalytics
from com.microsoft.spark.sqlanalytics.Constants import Constants


StatementMeta(SynapseSpark, 6, 11, Finished, Available, Finished, False)

In [ ]:
# Check your current Spark session context
print(f"Spark version  : {spark.version}")
print(f"App ID         : {sc.applicationId}")
print(f"App Name       : {sc.appName}")

# Check executor count — should be > 0 if pool is running
print(f"Executors      : {sc._jsc.sc().getExecutorMemoryStatus().size()}")


In [ ]:
from azure.identity import ClientSecretCredential
from azure.ai.textanalytics import TextAnalyticsClient
from openai import AzureOpenAI

# ── Config ────────────────────────────────────────────────────────────────
TENANT_ID         = # ← Replace with your tenant ID
CLIENT_ID         = # ← Replace with your service principal app ID
CLIENT_SECRET     = # Replace with your service principal client secret

LANGUAGE_ENDPOINT = # Replace with your Language Service endpoint "https://<your-language-service-name>.cognitiveservices.azure.com/"
OPENAI_ENDPOINT   = # Replace with your OpenAI endpoint "https://<your-openai-service-name>.openai.azure.com/"
OPENAI_API_VERSION = "2024-12-01-preview"
OPENAI_DEPLOYMENT  = "gpt-4o"


# ── Authenticate ──────────────────────────────────────────────────────────
try:
    credential = ClientSecretCredential(
        tenant_id     = TENANT_ID,
        client_id     = CLIENT_ID,
        client_secret = CLIENT_SECRET
    )

    # ── Language Client ───────────────────────────────────────────────────
    lang_client = TextAnalyticsClient(
        endpoint   = LANGUAGE_ENDPOINT,
        credential = credential
    )

    # ── OpenAI Client (custom subdomain endpoint supports Entra token auth) ──
    oai_token = credential.get_token("https://cognitiveservices.azure.com/.default")
    
    oai_client = AzureOpenAI(
        azure_endpoint = OPENAI_ENDPOINT,
        azure_ad_token = oai_token.token,
        api_version    = OPENAI_API_VERSION
    )

    print("✅ Service Principal authenticated successfully.")
    print(f"   Token expires at : {oai_token.expires_on}")
    print("✅ Language client  : ready")
    print("✅ OpenAI client    : ready")
    print(f"   OpenAI endpoint  : {OPENAI_ENDPOINT}")
    print(f"   OpenAI deployment: {OPENAI_DEPLOYMENT}")

except Exception as e:
    print(f"❌ Authentication failed: {e}")

In [8]:
test_docs = [
    "I absolutely love flying with this airline, the crew was amazing!",
    "The flight was delayed 3 hours and the seats were uncomfortable.",
    "It was okay, nothing special but nothing terrible either."
]

try:
    results = lang_client.analyze_sentiment(test_docs, language="en")
    print("✅ Sentiment analysis successful!\n")
    for i, result in enumerate(results):
        if not result.is_error:
            print(f"  Doc {i+1}: [{result.sentiment.upper():8}]"
                  f" | Pos: {result.confidence_scores.positive:.2f}"
                  f" | Neu: {result.confidence_scores.neutral:.2f}"
                  f" | Neg: {result.confidence_scores.negative:.2f}")
        else:
            print(f"  Doc {i+1} ERROR: {result.error.code} - {result.error.message}")

except Exception as e:
    print(f"❌ API call failed: {e}")

StatementMeta(SynapseSpark, 6, 17, Finished, Available, Finished, False)

✅ Sentiment analysis successful!

  Doc 1: [POSITIVE] | Pos: 1.00 | Neu: 0.00 | Neg: 0.00
  Doc 2: [NEGATIVE] | Pos: 0.00 | Neu: 0.00 | Neg: 1.00
  Doc 3: [NEUTRAL ] | Pos: 0.02 | Neu: 0.97 | Neg: 0.01


In [9]:
# OpenAI test using Entra token on custom-subdomain endpoint
from openai import AzureOpenAI
from azure.identity import ClientSecretCredential, get_bearer_token_provider

credential = ClientSecretCredential(
    tenant_id     = TENANT_ID,
    client_id     = CLIENT_ID,
    client_secret = CLIENT_SECRET
)

token_provider = get_bearer_token_provider(
    credential,
    "https://cognitiveservices.azure.com/.default"
)

client = AzureOpenAI(
    azure_endpoint          = OPENAI_ENDPOINT,
    azure_ad_token_provider = token_provider,
    api_version             = OPENAI_API_VERSION
)

# ── Test ──────────────────────────────────────────────────────────────────
try:
    response = client.chat.completions.create(
        model       = OPENAI_DEPLOYMENT,
        messages    = [{"role": "user", "content": "Say hello in one word."}],
        max_tokens  = 10,
        temperature = 0.3
    )
    print(f"✅ OpenAI test call succeeded: {response.choices[0].message.content.strip()}")
except Exception as e:
    print(f"❌ OpenAI test call failed: {e}")

StatementMeta(SynapseSpark, 6, 18, Finished, Available, Finished, False)

✅ OpenAI test call succeeded: Hello!


In [ ]:
import requests

tests = [
    ("OpenAI custom subdomain (token auth)", OPENAI_ENDPOINT),
    ("Language resource", LANGUAGE_ENDPOINT),
    ("Regional endpoint (reference only)", "https://centralus.api.cognitive.microsoft.com/")
]

for name, url in tests:
    try:
        r = requests.get(url, timeout=5)
        print(f"✅ Reachable : {name} — {url} — HTTP {r.status_code}")
    except requests.exceptions.Timeout:
        print(f"❌ Timeout   : {name} — {url}")
    except requests.exceptions.ConnectionError:
        print(f"❌ Blocked   : {name} — {url}")

In [ ]:
try:
    test_response = oai_client.chat.completions.create(
        model       = OPENAI_DEPLOYMENT,
        messages    = [{"role": "user", "content": "Say hello in one word."}],
        max_tokens  = 10,
        temperature = 0.3
    )
    print(f"✅ OpenAI test call succeeded: {test_response.choices[0].message.content.strip()}")
except Exception as e:
    print(f"❌ OpenAI test call failed: {e}")

In [ ]:
from pyspark.sql.functions import col
import com.microsoft.spark.sqlanalytics
from com.microsoft.spark.sqlanalytics.Constants import Constants

# ── Load FAComments ───────────────────────────────────────────────────────
df = spark.read.synapsesql("<your-database>.dbo.FAComments") \
    .filter("Comments IS NOT NULL AND TRIM(Comments) != ''") \
    .select("AssociateId", "FlightDate", "Comments")

print(f"✅ Rows to process: {df.count()}")


In [13]:
# Sentiment Analysis w/Opinions (driver-side to avoid executor package issues)
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType
from azure.identity import ClientSecretCredential
from azure.ai.textanalytics import TextAnalyticsClient

BATCH_SIZE = 10

# ── Output Schema ─────────────────────────────────────────────────────────
result_schema = StructType([
    StructField("AssociateId",    IntegerType(), True),
    StructField("FlightDate",     StringType(),  True),
    StructField("Comments",       StringType(),  True),
    StructField("Sentiment",      StringType(),  True),
    StructField("Score_Positive", FloatType(),   True),
    StructField("Score_Neutral",  FloatType(),   True),
    StructField("Score_Negative", FloatType(),   True),
    StructField("Opinions",       StringType(),  True),
])

# ── Initialize client on driver ───────────────────────────────────────────
credential = ClientSecretCredential(
    tenant_id     = TENANT_ID,
    client_id     = CLIENT_ID,
    client_secret = CLIENT_SECRET
)
lang_client = TextAnalyticsClient(
    endpoint   = LANGUAGE_ENDPOINT,
    credential = credential
)

# ── Driver-side processing (no Spark worker imports required) ────────────
rows = list(df.select("AssociateId", "FlightDate", "Comments").toLocalIterator())
output = []

for i in range(0, len(rows), BATCH_SIZE):
    batch = rows[i:i + BATCH_SIZE]
    texts = [str(r["Comments"]) for r in batch]

    try:
        results = lang_client.analyze_sentiment(
            texts,
            language="en",
            show_opinion_mining=True
        )
        for row, result in zip(batch, results):
            if not result.is_error:
                aspects = []
                for sentence in result.sentences:
                    for mined_opinion in sentence.mined_opinions:
                        target = mined_opinion.target
                        for assessment in mined_opinion.assessments:
                            aspects.append(
                                f"{target.text}:{assessment.text}"
                                f"({assessment.sentiment})"
                            )
                output.append((
                    int(row["AssociateId"]),
                    str(row["FlightDate"]),
                    str(row["Comments"]),
                    result.sentiment,
                    float(result.confidence_scores.positive),
                    float(result.confidence_scores.neutral),
                    float(result.confidence_scores.negative),
                    "|".join(aspects) if aspects else ""
                ))
            else:
                output.append((
                    int(row["AssociateId"]),
                    str(row["FlightDate"]),
                    str(row["Comments"]),
                    f"ERROR: {result.error.code}",
                    0.0, 0.0, 0.0, ""
                ))
    except Exception as e:
        for row in batch:
            output.append((
                int(row["AssociateId"]),
                str(row["FlightDate"]),
                str(row["Comments"]),
                f"EXCEPTION: {str(e)}",
                0.0, 0.0, 0.0, ""
            ))

results_df = spark.createDataFrame(output, schema=result_schema)
results_df.show(10, truncate=80)
print(f"✅ Total rows processed: {results_df.count()}")

StatementMeta(SynapseSpark, 6, 22, Finished, Available, Finished, False)

+-----------+----------+--------------------------------------------------------------------------------+---------+--------------+-------------+--------------+--------------------------------------------------------------------------------+
|AssociateId|FlightDate|                                                                        Comments|Sentiment|Score_Positive|Score_Neutral|Score_Negative|                                                                        Opinions|
+-----------+----------+--------------------------------------------------------------------------------+---------+--------------+-------------+--------------+--------------------------------------------------------------------------------+
|     613167|   2025-05|She was bubbly and positive! A pleasure. Kristen had a very positive attitude...|    mixed|          0.65|         0.02|          0.33|customer service:great(positive)|announcements:Too many(negative)|PA:loud(neg...|
|     611920|   2025-05|She was very

In [ ]:
# Uses extractive summarization to pull most representative sentence from each comment. More Deterministic than abstractive summarization.
from azure.ai.textanalytics import ExtractiveSummaryAction

# ── Collect results_df to driver ──────────────────────────────────────────
rows_collected = results_df.collect()
print(f"📝 Summarizing {len(rows_collected)} rows using Language Service...")

# ── Helper — Extractive Summary in batches of 10 ─────────────────────────
def summarize_batch(comments):
    """
    Takes a list of up to 10 comments, returns a list of summary strings.
    Uses extractive summarization — pulls the most representative sentence.
    """
    try:
        poller = lang_client.begin_analyze_actions(
            comments,
            actions=[ExtractiveSummaryAction(max_sentence_count=1)],
            language="en"
        )
        summaries = []
        for page in poller.result():
            for doc in page:
                if not doc.is_error:
                    summaries.append(
                        " ".join(s.text for s in doc.sentences)
                    )
                else:
                    summaries.append(f"ERROR: {doc.error.code}")
        return summaries
    except Exception as e:
        return [f"EXCEPTION: {str(e)}"] * len(comments)


# ── Process in batches of 10 ──────────────────────────────────────────────
final_rows = []
SUMMARY_BATCH_SIZE = 10

for i in range(0, len(rows_collected), SUMMARY_BATCH_SIZE):
    batch    = rows_collected[i:i + SUMMARY_BATCH_SIZE]
    comments = [str(row["Comments"]) for row in batch]
    summaries = summarize_batch(comments)

    for row, summary in zip(batch, summaries):
        final_rows.append((
            row["AssociateId"],
            row["FlightDate"],
            row["Sentiment"],
            row["Score_Positive"],
            row["Score_Neutral"],
            row["Score_Negative"],
            row["Opinions"],
            summary
        ))

print(f"✅ Summarization complete. {len(final_rows)} rows processed.")

# ── Rebuild as Spark DataFrame ────────────────────────────────────────────
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

final_schema = StructType([
    StructField("AssociateId",    IntegerType(), True),
    StructField("FlightDate",     StringType(),  True),
    StructField("Sentiment",      StringType(),  True),
    StructField("Score_Positive", FloatType(),   True),
    StructField("Score_Neutral",  FloatType(),   True),
    StructField("Score_Negative", FloatType(),   True),
    StructField("Opinions",       StringType(),  True),
    StructField("Summary",        StringType(),  True),
])

final_df = spark.createDataFrame(final_rows, schema=final_schema)
final_df.show(10, truncate=80)
print(f"✅ Total rows with summaries: {final_df.count()}")

StatementMeta(SynapseSpark, 6, 23, Finished, Available, Finished, False)

📝 Summarizing 23 rows using Language Service...
✅ Summarization complete. 23 rows processed.
+-----------+----------+---------+--------------+-------------+--------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|AssociateId|FlightDate|Sentiment|Score_Positive|Score_Neutral|Score_Negative|                                                                        Opinions|                                                                         Summary|
+-----------+----------+---------+--------------+-------------+--------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|     613167|   2025-05|    mixed|          0.65|         0.02|          0.33|customer service:great(positive)|announcements:Too many(negative)|PA:loud(neg...|                Kristen h

In [ ]:
# Uses abstractive summarization via OpenAI to generate a concise summary sentence for each comment. More flexible but less deterministic than extractive summarization.
# ── Collect results_df to driver ──────────────────────────────────────────
rows_collected = results_df.collect()
print(f"📝 Summarizing {len(rows_collected)} rows on driver...")

# ── Summarize each row using oai_client ───────────────────────────────────
def generate_summary(comment, opinions):
    formatted_opinions = ", ".join([
        o.replace(":", ": ").replace("(", " (")
        for o in opinions.split("|") if o
    ]) if opinions else "No specific opinions extracted."

    prompt = f"""You are analyzing airline passenger survey comments.
Given the passenger comment and extracted opinion aspects below,
write a single concise sentence (max 25 words) summarizing the
passenger's overall experience.

Comment  : {comment}
Opinions : {formatted_opinions}

Rules:
- One sentence only
- Be specific — mention the actual aspects (crew, seat, delay, etc.)
- Use neutral, professional language
- Do not start with "The passenger"

Summary:"""

    try:
        response = oai_client.chat.completions.create(
            model       = OPENAI_DEPLOYMENT,
            messages    = [{"role": "user", "content": prompt}],
            max_tokens  = 60,
            temperature = 0.3
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"SUMMARY_ERROR: {str(e)}"

# ── Build final output with summaries ─────────────────────────────────────
final_rows = []
for row in rows_collected:
    summary = generate_summary(row["Comments"], row["Opinions"]) \
              if "ERROR" not in row["Sentiment"] else ""
    final_rows.append((
        row["AssociateId"],
        row["FlightDate"],
        row["Sentiment"],
        row["Score_Positive"],
        row["Score_Neutral"],
        row["Score_Negative"],
        row["Opinions"],
        summary
    ))

# ── Rebuild as Spark DataFrame ────────────────────────────────────────────
final_schema = StructType([
    StructField("AssociateId",    IntegerType(), True),
    StructField("FlightDate",     StringType(),  True),
    StructField("Sentiment",      StringType(),  True),
    StructField("Score_Positive", FloatType(),   True),
    StructField("Score_Neutral",  FloatType(),   True),
    StructField("Score_Negative", FloatType(),   True),
    StructField("Opinions",       StringType(),  True),
    StructField("Summary",        StringType(),  True),
])

final_df = spark.createDataFrame(final_rows, schema=final_schema)
final_df.show(10, truncate=80)
print(f"✅ Total rows with summaries: {final_df.count()}")

StatementMeta(SynapseSpark, 6, 24, Finished, Available, Finished, False)

📝 Summarizing 23 rows on driver...
+-----------+----------+---------+--------------+-------------+--------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|AssociateId|FlightDate|Sentiment|Score_Positive|Score_Neutral|Score_Negative|                                                                        Opinions|                                                                         Summary|
+-----------+----------+---------+--------------+-------------+--------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|     613167|   2025-05|    mixed|          0.65|         0.02|          0.33|customer service:great(positive)|announcements:Too many(negative)|PA:loud(neg...|The passenger appreciated Kristen's excellent customer service but found the ...|
|